<a href="https://colab.research.google.com/github/gabriellbragaa/Computer_Vision_Projs/blob/main/labResolu%C3%A7%C3%A3o07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


01) Usando o classificador HaarCascade, crie um programa que realize a detecção de rosto
utilizando imagens. Teste com imagens de diferentes resoluções e crie uma tabela com os
tempos de execução em cada caso.





In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image, Markdown
import io
import cv2
import numpy as np
import time
import pandas as pd


face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml') # dependencias para face
eyes_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml') # dependencias para olhos
mouth_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml') # dependecias para boca

if face_cascade.empty():
    print("Erro ao carregar o classificador cascade para rosto.")
if eyes_cascade.empty():
    print("Erro ao carregar o classificador cascade para olhos.")
if mouth_cascade.empty():
    print("Erro ao carregar o classificador cascade para boca.")

# Função para detectar os rostos
def detect_faces(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(25, 25))
    return faces

def detect_eyes_mouth(img, faces):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    face_details = []

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y + h, x:x + w]
        roi_color = img[y:y + h, x:x + w]

        eyes = eyes_cascade.detectMultiScale(roi_gray,scaleFactor=1.2, minNeighbors=10, minSize=(5, 8) )
        mouth = mouth_cascade.detectMultiScale(roi_gray, scaleFactor=1.7, minNeighbors=30, minSize=(25, 25))

        # scaleFactor =  Este parâmetro especifica o quanto o tamanho da imagem é reduzido em cada escala de imagem.

        # minNeighbors =  Este parâmetro especifica quantos vizinhos cada retângulo candidato deve ter para retê-lo.
        ## Um valor mais alto resulta em menos detecções, mas com maior confiança (reduzindo falsos positivos e múltiplas detecções do mesmo objeto).

        # minSize = Este parâmetro especifica o tamanho mínimo possível do objeto.
        ## Objetos menores que isso são ignorados. Neste caso, a boca detectada deve ter pelo menos 22x25 pixels.

        detected_eyes = []
        for (ex, ey, ew, eh) in eyes:
            eye_type = 'unknown'
            if ex + ew/2 < w/2:
                eye_type = 'left'
            else:
                eye_type = 'right'
            detected_eyes.append({'coords': (x + ex, y + ey, ew, eh), 'type': eye_type})

        detected_mouth = [(x + mx, y + my, mw, mh) for (mx, my, mw, mh) in mouth]

        face_details.append({
            'face': (x, y, w, h),
            'eyes': detected_eyes,
            'mouth': detected_mouth
        })

    return face_details


uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False
)


output = widgets.Output()


execution_times_data = []


def handle_upload(change):
    with output:
        output.clear_output()
        if not uploader.value:
            print("Nenhum arquivo enviado.")
            return

        uploaded_file_info = list(uploader.value.values())[0]
        content = uploaded_file_info['content']
        file_name = uploaded_file_info['metadata']['name']

        try:
            image_array = np.frombuffer(content, np.uint8)
            img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

            if img is None:
                 print(f"Erro: Não foi possível decodificar a imagem {file_name}")
                 return

            start_time = time.time()
            processed_image = img.copy()

            faces = detect_faces(processed_image)
            face_details = detect_eyes_mouth(processed_image, faces)

            # detections
            for detail in face_details:
                # rosto
                (x, y, w, h) = detail['face']
                cv2.rectangle(processed_image, (x, y), (x + w, y + h), (255, 0, 0), 2)

                # olhos
                for eye in detail['eyes']:
                    (ex, ey, ew, eh) = eye['coords']

                    color = (0, 255, 0)
                    cv2.rectangle(processed_image, (ex, ey), (ex + ew, ey + eh), color, 2)

                # boca
                for (mx, my, mw, mh) in detail['mouth']:
                    cv2.rectangle(processed_image, (mx, my), (mx + mw, my + mh), (255, 255, 0), 2)


            end_time = time.time()
            exec_time = end_time - start_time

            if processed_image is not None:
                is_success, im_buf_arr = cv2.imencode(".png", processed_image)
                if is_success:
                    byte_im = im_buf_arr.tobytes()
                    print(f"Tempo de processamento: {exec_time:.4f} segundos")
                    display(Image(byte_im))

                    execution_times_data.append({'Image': file_name, 'Execution Time (s)': f"{exec_time:.4f}"})

                else:
                    print("Erro ao converter a imagem processada para bytes.")
            else:
                print("O processamento da imagem falhou.")

        except Exception as e:
            print(f"Ocorreu um erro: {e}")

uploader.observe(handle_upload, names='value')

# Create a function to display the execution times table
def display_execution_times_table():

    if not execution_times_data:
        display(Markdown("### Nenhuma imagem processada ainda para exibir os tempos de execução."))
        return

    df_times = pd.DataFrame(execution_times_data)
    display(Markdown("### Tempos de Execução do Processamento de Imagem"))
    display(df_times)

# Optional: Display a button to trigger the table display
display_table_button = widgets.Button(description="Mostrar Tabela de Tempos de Execução")

def on_button_click(b):
    with output:
        output.clear_output()
        display_execution_times_table()

display_table_button.on_click(on_button_click)

display(uploader, output, display_table_button)

print("Widget de upload, área de saída e botão da tabela exibidos. Pronto para envio de imagem.")

FileUpload(value={}, accept='image/*', description='Upload')

Output()

Button(description='Mostrar Tabela de Tempos de Execução', style=ButtonStyle())

Widget de upload, área de saída e botão da tabela exibidos. Pronto para envio de imagem.


Repita a questão anterior mas detecte também a boca e os olhos, diferenciando entre
olho direito e olho esquerdo.
– Em ambas as questões inclua imagens de rostos com as seguintes características:
a) Vistos frontal e lateralmente;
b) Utilizando acessórios (chapéu, óculos, máscaras, etc);
c) Com e Sem barba, cabelos curtos/compridos/sem cabelo;
d) De pessoas de diferentes etnias.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, Image, Markdown
import io
import cv2
import numpy as np
import time
import pandas as pd

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
eyes_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
mouth_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_smile.xml') # Using smile for mouth detection


if face_cascade.empty():
    print("Erro ao carregar o classificador cascade para rosto.")
if eyes_cascade.empty():
    print("Erro ao carregar o classificador cascade para olhos.")
if mouth_cascade.empty():
    print("Erro ao carregar o classificador cascade para boca.")


def detect_faces(img):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
    return faces


def detect_eyes_mouth(img, faces):

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    face_details = []

    for (x, y, w, h) in faces:
        roi_gray = gray[y:y + h, x:x + w]
        roi_color = img[y:y + h, x:x + w]


        eyes = eyes_cascade.detectMultiScale(roi_gray, scaleFactor=1.1, minNeighbors=10, minSize=(20, 20))

        mouth = mouth_cascade.detectMultiScale(roi_gray, scaleFactor=1.7, minNeighbors=22, minSize=(25, 25))

        detected_eyes = []

        for (ex, ey, ew, eh) in eyes:
            eye_type = 'unknown'

            if ex + ew/2 < w/2:
                eye_type = 'left'
            else:
                eye_type = 'right'

            detected_eyes.append({'coords': (x + ex, y + ey, ew, eh), 'type': eye_type})


        detected_mouth = [(x + mx, y + my, mw, mh) for (mx, my, mw, mh) in mouth]

        face_details.append({
            'face': (x, y, w, h),
            'eyes': detected_eyes,
            'mouth': detected_mouth
        })

    return face_details


uploader = widgets.FileUpload(
    accept='image/*',
    multiple=False
)


output = widgets.Output()

execution_times_data = []


def handle_upload(change):
    """
    Handles the file upload event, processes the image, and displays results.
    """
    with output:
        output.clear_output()

        if not uploader.value:
            print("Nenhum arquivo enviado.")
            return


        uploaded_file_info = list(uploader.value.values())[0]
        content = uploaded_file_info['content']
        file_name = uploaded_file_info['metadata']['name']

        try:

            image_array = np.frombuffer(content, np.uint8)
            img = cv2.imdecode(image_array, cv2.IMREAD_COLOR)

            if img is None:
                 print(f"Erro: Não foi possível decodificar a imagem {file_name}")
                 return


            start_time = time.time()
            processed_image = img.copy()

            faces = detect_faces(processed_image)
            face_details = detect_eyes_mouth(processed_image, faces)

            for detail in face_details:

                (x, y, w, h) = detail['face']
                cv2.rectangle(processed_image, (x, y), (x + w, y + h), (255, 0, 0), 2)


                for eye in detail['eyes']:
                    (ex, ey, ew, eh) = eye['coords']
                    color = (0, 255, 0) # Default Green
                    if eye['type'] == 'left':
                         color = (0, 255, 255) # Yellow for left eye
                    elif eye['type'] == 'right':
                         color = (0, 0, 255) # Red for right eye

                    cv2.rectangle(processed_image, (ex, ey), (ex + ew, ey + eh), color, 2)


                for (mx, my, mw, mh) in detail['mouth']:
                    cv2.rectangle(processed_image, (mx, my), (mx + mw, my + mh), (255, 255, 0), 2)


            end_time = time.time()
            exec_time = end_time - start_time


            if processed_image is not None:
                is_success, im_buf_arr = cv2.imencode(".png", processed_image)
                if is_success:
                    byte_im = im_buf_arr.tobytes()
                    print(f"Tempo de processamento: {exec_time:.4f} segundos")
                    display(Image(byte_im))


                    execution_times_data.append({'Image': file_name, 'Execution Time (s)': f"{exec_time:.4f}"})

                else:
                    print("Erro ao converter a imagem processada para bytes.")
            else:
                print("O processamento da imagem falhou.")

        except Exception as e:
            print(f"Ocorreu um erro: {e}")


uploader.observe(handle_upload, names='value')


def display_execution_times_table():

    if not execution_times_data:
        display(Markdown("### Nenhuma imagem processada ainda para exibir os tempos de execução."))
        return

    df_times = pd.DataFrame(execution_times_data)
    display(Markdown("### Tempos de Execução do Processamento de Imagem"))
    display(df_times)


display_table_button = widgets.Button(description="Mostrar Tabela de Tempos de Execução")


def on_button_click(b):
    with output:
        output.clear_output()
        display_execution_times_table()


display_table_button.on_click(on_button_click)


display(uploader, output, display_table_button)

print("Widget de upload, área de saída e botão da tabela exibidos. Pronto para envio de imagem.")

FileUpload(value={}, accept='image/*', description='Upload')

Output()

Button(description='Mostrar Tabela de Tempos de Execução', style=ButtonStyle())

Widget de upload, área de saída e botão da tabela exibidos. Pronto para envio de imagem.
